##### ARTI 560 - Computer Vision

## Action Recognition - Exercise

### Objective

In this exercise, you will train a deep learning model to recognize three specific human actions using the [UCF11 (YouTube Action) dataset](https://www.crcv.ucf.edu/data/UCF_YouTube_Action.php) and validate the model's real-world performance using external video data.

*[Note: This notebook is based on [this](https://github.com/Sumaya2026/learnopencv/tree/master/Optical-Flow-Estimation-using-Deep-Learning-RAFT) GitHub Repository by LearnOpenCV]*


#### Tasks

- Choose **three classes** from the UCF11 dataset (e.g., Basketball Shooting, Biking, Tennis Swinging, etc.).
- Preprocess the dataset.
- Split the data into training and testing.
- Create and train the model.
- Save the trained model .
    **Important Note**: The final trained model must be saved with a filename that includes your name. This is a mandatory step for the submission.
    ```
    # Example Code
    student_name = "YourName" # Replace with your actual name
    save_path = f"{student_name}_ucf11_model.h5"
    model.save(save_path)
    print(f"Model saved as {save_path}")
    ```
- Validate the model on 3 Youtube videos, each clearly showing one of your three chosen action classes.


In [3]:
import os
import cv2
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split


IMAGE_HEIGHT , IMAGE_WIDTH = 64, 64
SEQUENCE_LENGTH = 20
DATASET_DIR = r"C:\Users\djood\Downloads\UCF50"
CLASSES_LIST = ["Basketball", "Biking", "TennisSwing"]


student_name = "Joud Alahmari"

In [ ]:
def frames_extraction(video_path):
    frames_list = []
    video_reader = cv2.VideoCapture(video_path)
    video_frames_count = int(video_reader.get(cv2.CAP_PROP_FRAME_COUNT))
    skip_frames_window = max(int(video_frames_count / SEQUENCE_LENGTH), 1)

    for frame_counter in range(SEQUENCE_LENGTH):
        video_reader.set(cv2.CAP_PROP_POS_FRAMES, frame_counter * skip_frames_window)
        success, frame = video_reader.read() 
        if not success:
            break
        resized_frame = cv2.resize(frame, (IMAGE_HEIGHT, IMAGE_WIDTH))
        normalized_frame = resized_frame / 255
        frames_list.append(normalized_frame)
    
    video_reader.release()
    return frames_list

if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(f"Dataset directory not found: {DATASET_DIR}")

nested_dataset_dir = os.path.join(DATASET_DIR, os.path.basename(DATASET_DIR))
if os.path.isdir(nested_dataset_dir):
    print("Detected nested dataset root. Switching to:", nested_dataset_dir)
    DATASET_DIR = nested_dataset_dir

print("Using dataset directory:", DATASET_DIR)
print("Classes:", CLASSES_LIST)


def get_video_file_paths(class_dir):
    valid_ext = {".avi", ".mp4", ".mpeg", ".mpg", ".mov", ".wmv"}
    return [
        os.path.join(class_dir, f)
        for f in sorted(os.listdir(class_dir))
        if os.path.splitext(f)[1].lower() in valid_ext
    ]


def create_dataset():
    features = []
    labels = []
    
    for class_index, class_name in enumerate(CLASSES_LIST):
        class_dir = os.path.join(DATASET_DIR, class_name)
        if not os.path.isdir(class_dir):
            raise FileNotFoundError(f"Class directory not found: {class_dir}")
        print(f'Extracting data for class: {class_name}')

        for video_file_path in get_video_file_paths(class_dir):
            frames = frames_extraction(video_file_path)
            if len(frames) == SEQUENCE_LENGTH:
                features.append(frames)
                labels.append(class_index)
            else:
                print(f"Skipping {video_file_path} ({len(frames)} frames)")

    return np.asarray(features, dtype=np.float32), np.array(labels, dtype=np.int32)

features, labels = create_dataset()
print("Dataset shape:", features.shape, labels.shape)

one_hot_encoded_labels = tf.keras.utils.to_categorical(labels, num_classes=len(CLASSES_LIST))
X_train, X_test, y_train, y_test = train_test_split(features, one_hot_encoded_labels, test_size=0.2, shuffle=True, random_state=42)
print("Train/Test split:", X_train.shape, X_test.shape, y_train.shape, y_test.shape)

model = Sequential([
    Conv3D(32, kernel_size=(3, 3, 3), activation='relu', input_shape=(SEQUENCE_LENGTH, IMAGE_HEIGHT, IMAGE_WIDTH, 3)),
    MaxPool3D(pool_size=(1, 2, 2)),
    BatchNormalization(),
    Conv3D(64, kernel_size=(3, 3, 3), activation='relu'),
    MaxPool3D(pool_size=(2, 2, 2)),
    BatchNormalization(),
    Conv3D(128, kernel_size=(3, 3, 3), activation='relu'),
    MaxPool3D(pool_size=(2, 2, 2)),
    BatchNormalization(),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(len(CLASSES_LIST), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

callbacks = [EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)]

history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=20,
    batch_size=8,
    callbacks=callbacks,
    verbose=2
)

eval_results = model.evaluate(X_test, y_test, verbose=2)
print(f"Test loss: {eval_results[0]:.4f}, Test accuracy: {eval_results[1]:.4f}")

save_path_ucf11 = f"{student_name.replace(' ', '_')}_ucf11_model.h5"
model.save(save_path_ucf11)
print(f"Model saved as {save_path_ucf11}")

save_path_ucf50 = f"{student_name.replace(' ', '_')}_ucf50_model.h5"
model.save(save_path_ucf50)
print(f"Also saved a second copy as {save_path_ucf50}")


def predict_action(video_path):
    frames = frames_extraction(video_path)
    if len(frames) != SEQUENCE_LENGTH:
        raise ValueError(f"Video must yield {SEQUENCE_LENGTH} frames, got {len(frames)}")
    X = np.expand_dims(frames, axis=0)
    probs = model.predict(X)[0]
    predicted_index = np.argmax(probs)
    return CLASSES_LIST[predicted_index], float(probs[predicted_index])





Detected nested dataset root. Switching to: C:\Users\djood\Downloads\UCF50\UCF50
Using dataset directory: C:\Users\djood\Downloads\UCF50\UCF50
Classes: ['Basketball', 'Biking', 'TennisSwing']
Extracting data for class: Basketball
Extracting data for class: Biking
Extracting data for class: TennisSwing
Dataset shape: (449, 20, 64, 64, 3) (449,)
Train/Test split: (359, 20, 64, 64, 3) (90, 20, 64, 64, 3) (359, 3) (90, 3)


c:\Users\djood\anaconda3\envs\cv_lab\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv3d (Conv3D)                 │ (None, 18, 62, 62, 32) │         2,624 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d (MaxPooling3D)    │ (None, 18, 31, 31, 32) │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 18, 31, 31, 32) │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_1 (Conv3D)               │ (None, 16, 29, 29, 64) │        55,360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_1 (MaxPooling3D)  │ (None, 8, 14, 14, 64)  │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 8, 14, 14, 64)  │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3d_2 (Conv3D)               │ (None, 6, 12, 12, 128) │       221,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling3d_2 (MaxPooling3D)  │ (None, 3, 6, 6, 128)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 3, 6, 6, 128)   │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 13824)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     3,539,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,820,163 (14.57 MB)

 Trainable params: 3,819,715 (14.57 MB)

 Non-trainable params: 448 (1.75 KB)

Epoch 1/20
36/36 - 12s - 322ms/step - accuracy: 0.6411 - loss: 4.6262 - val_accuracy: 0.3194 - val_loss: 5.9065
Epoch 2/20
36/36 - 10s - 269ms/step - accuracy: 0.8014 - loss: 3.0815 - val_accuracy: 0.5278 - val_loss: 5.7669
Epoch 3/20
36/36 - 10s - 274ms/step - accuracy: 0.7666 - loss: 2.5368 - val_accuracy: 0.4028 - val_loss: 6.9200
Epoch 4/20
36/36 - 10s - 280ms/step - accuracy: 0.8641 - loss: 1.6375 - val_accuracy: 0.3750 - val_loss: 7.1418
Epoch 5/20
36/36 - 11s - 294ms/step - accuracy: 0.8606 - loss: 1.6165 - val_accuracy: 0.6806 - val_loss: 2.0738
Epoch 6/20
36/36 - 11s - 293ms/step - accuracy: 0.8537 - loss: 1.4415 - val_accuracy: 0.5694 - val_loss: 4.7480
Epoch 7/20
36/36 - 10s - 276ms/step - accuracy: 0.8850 - loss: 1.2969 - val_accuracy: 0.6389 - val_loss: 3.5440
Epoch 8/20
36/36 - 10s - 274ms/step - accuracy: 0.9164 - loss: 0.9063 - val_accuracy: 0.7500 - val_loss: 2.6988
3/3 - 1s - 339ms/step - accuracy: 0.6556 - loss: 3.3219


Test loss: 3.3219, Test accuracy: 0.6556
Model saved as Joud_Alahmari_ucf50_model.h5


In [15]:
# Validation on three YouTube videos

# YouTube video URLs for each class
youtube_urls = {
    "Basketball": "https://youtu.be/SyvuSxCyfi0",  # Basketball shooting
    "Biking": "https://youtu.be/GERp-h-BnD8",  # Biking
    "TennisSwing": "https://youtu.be/JQ9Fb5mfJu4",  # Tennis swing
}

validation_dir = r"C:\Users\djood\Downloads\validation_videos"
os.makedirs(validation_dir, exist_ok=True)

# Try using yt-dlp for reliable YouTube downloads
try:
    import yt_dlp
except ImportError:
    print("Installing yt-dlp for YouTube downloads...")
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yt-dlp", "-q"])
    import yt_dlp

validation_videos = []

for class_name, url in youtube_urls.items():
    output_path = os.path.join(validation_dir, f"{class_name}.mp4")
    
    if os.path.exists(output_path):
        print(f"Using existing video: {output_path}")
        validation_videos.append(output_path)
        continue
    
    try:
        print(f"Downloading {class_name} from YouTube...")
        ydl_opts = {
            'format': 'best[ext=mp4]/best',
            'outtmpl': os.path.join(validation_dir, f"{class_name}"),
            'quiet': False,
            'no_warnings': False,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(url, download=True)
            downloaded_file = ydl.prepare_filename(info)
            if os.path.exists(downloaded_file):
                validation_videos.append(downloaded_file)
                print(f"✓ Successfully downloaded: {downloaded_file}")
    except Exception as ex:
        print(f"✗ Failed to download {class_name} from YouTube: {str(ex)[:100]}")

# Fallback: if YouTube download fails, use videos from the dataset
if not validation_videos:
    print("\n⚠ YouTube download failed. Using videos from UCF50 dataset for validation...")
    for class_name in CLASSES_LIST:
        class_dir = os.path.join(DATASET_DIR, class_name)
        if os.path.isdir(class_dir):
            video_files = [f for f in os.listdir(class_dir) 
                          if os.path.splitext(f)[1].lower() in {".avi", ".mp4", ".mpeg", ".mpg", ".mov", ".wmv"}]
            if video_files:
                video_path = os.path.join(class_dir, video_files[0])
                validation_videos.append(video_path)
                print(f"Using dataset video for {class_name}: {video_files[0]}")

if not validation_videos:
    raise RuntimeError("Could not find or download any validation videos.")

print("\n" + "="*80)
print("VALIDATION RESULTS")
print("="*80 + "\n")

for video_path in validation_videos:
    try:
        action, confidence = predict_action(video_path)
        video_name = os.path.basename(video_path)
        class_name = os.path.basename(video_path).split('.')[0]
        print(f"Video: {video_name}")
        print(f"  Predicted Action: {action} | Confidence: {confidence:.4f}\n")
    except Exception as ex:
        print(f"Error predicting for {video_path}: {ex}\n")

print("="*80)
print("Lab 07 - Action Recognition Exercise COMPLETE ✓")
print("="*80)

[youtube] Extracting URL: https://youtu.be/SyvuSxCyfi0
[youtube] SyvuSxCyfi0: Downloading webpage


[youtube] SyvuSxCyfi0: Downloading android vr player API JSON
[info] SyvuSxCyfi0: Downloading 1 format(s): 18
[download] C:\Users\djood\Downloads\validation_videos\Basketball has already been downloaded
[download] 100% of    8.30MiB
✓ Successfully downloaded: C:\Users\djood\Downloads\validation_videos\Basketball
[youtube] Extracting URL: https://youtu.be/GERp-h-BnD8
[youtube] GERp-h-BnD8: Downloading webpage


[youtube] GERp-h-BnD8: Downloading android vr player API JSON
[info] GERp-h-BnD8: Downloading 1 format(s): 18
[download] C:\Users\djood\Downloads\validation_videos\Biking has already been downloaded
[download] 100% of   11.28MiB
✓ Successfully downloaded: C:\Users\djood\Downloads\validation_videos\Biking
[youtube] Extracting URL: https://youtu.be/JQ9Fb5mfJu4
[youtube] JQ9Fb5mfJu4: Downloading webpage


[youtube] JQ9Fb5mfJu4: Downloading android vr player API JSON
[info] JQ9Fb5mfJu4: Downloading 1 format(s): 18
[download] C:\Users\djood\Downloads\validation_videos\TennisSwing has already been downloaded
[download] 100% of  614.43KiB
✓ Successfully downloaded: C:\Users\djood\Downloads\validation_videos\TennisSwing

VALIDATION RESULTS

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 112ms/step
Video: Basketball
  Predicted Action: Biking | Confidence: 1.0000

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
Video: Biking
  Predicted Action: Biking | Confidence: 0.9992

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step
Video: TennisSwing
  Predicted Action: TennisSwing | Confidence: 0.4919

Lab 07 - Action Recognition Exercise COMPLETE ✓
